## 导入包

In [1]:
import os
import datetime
import argparse
import time
import numpy as np
import torch as th
import torch.distributed as dist
import matplotlib.pyplot as plt
import gc
from pathlib import Path

from improved_diffusion import dist_util, logger
from improved_diffusion.dataset import get_dataloader, yield_dataloader, show_dataloader, show_samples
from improved_diffusion.resample import create_named_schedule_sampler
from improved_diffusion.script_util import (
    model_and_diffusion_defaults,
    create_model_and_diffusion,
    args_to_dict,
    add_dict_to_argparser,
    sample_filter,
)
from improved_diffusion.losses import compute_F1_score
from improved_diffusion.train_util import TrainLoop

## 参数设置

In [ ]:
def create_argparser():
    defaults = model_and_diffusion_defaults()
    
    my_config = dict(
        root_folder_data_train = Path('../datasets/PPD/train'),
        folder_Mo_train ='MAP_with_start_end',
        folder_P_train ='PATH_2PIXEL',
        num_images_train = 12000,
        root_folder_data_sample = Path('../datasets/PPD/test/UNSEEN'),
        folder_Mo_sample ='MAP_with_start_end',
        folder_P_sample ='PATH_2PIXEL',
        num_images_sample = 1200,
        batch_size=8,
        image_size=128,
        
        biased_initialization = 1e-3,   # 是否使用 偏置初始化-范式P2
        weight_path_similarity = 0,     # 路径相似度损失，要求恢复的图像和路径尽可能相似
        use_MFF_MAC = False,            # 是否使用 MFF_MAC
        Path_inverse = True,            # 路径是否反相，似乎 反相 的效果更好
        use_CFM = True,

        schedule_sampler="loss-second-moment",     
        lr=1e-4,                        # Adamw的学习率
        weight_decay=0.0,               # Adamw的权重衰减
        lr_anneal_steps=60000,
        microbatch=-1,                  # -1 disables microbatches
        ema_rate="0.999",               # comma-separated list of EMA values
        resume_checkpoint="",
        use_fp16=False,
        fp16_scale_growth=1e-3,
    )
    
    defaults.update(my_config)
    parser = argparse.ArgumentParser()
    add_dict_to_argparser(parser, defaults)
    
    return parser

## 定义实验矩阵

In [3]:
test_matrix = [
    {"use_MFF_MAC": True,  "Path_inverse": True},
    {"use_MFF_MAC": True,  "Path_inverse": False},
    {"use_MFF_MAC": False, "Path_inverse": True},
    {"use_MFF_MAC": False, "Path_inverse": False},
]

## 循环测试

In [4]:
for config in test_matrix:
    print("="*50)
    print(f"Starting Experiment: use_MFF_MAC={config['use_MFF_MAC']}, Path_inverse={config['Path_inverse']}")
    print("="*50)

    # 1. 获取默认参数并更新当前实验配置
    parser = create_argparser()
    args = parser.parse_args([])
    args.log_interval=args.lr_anneal_steps/20
    args.save_interval=args.lr_anneal_steps/20
    
    # 更新当前循环的特定参数
    args.use_MFF_MAC = config['use_MFF_MAC']
    args.Path_inverse = config['Path_inverse']

    # 2. 动态生成唯一的日志目录
    subdir = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S") + f'-P2_{args.biased_initialization}-path_simil{args.weight_path_similarity}-use_MFF_MAC-{args.use_MFF_MAC}-{args.folder_P_train}-path_inv_{args.Path_inverse}-img{args.image_size}-batch_{args.batch_size}-epochs{args.lr_anneal_steps}'
    log_dir = os.path.join(os.path.expanduser("~/DiffRP_IDDPM/my_model_checkpoints"), subdir)
    
    # 强制更新环境变量（确保 logger 使用新的路径）
    os.environ["OPENAI_LOGDIR"] = log_dir
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    
    # 3. 重新配置 logger
    dist_util.setup_dist()  # 设置分布式训练环境，确保在多GPU或多节点环境下正确地初始化通信和资源分配
    logger.configure(dir=log_dir) 
    
    # 4. 创建模型和扩散过程
    model, diffusion = create_model_and_diffusion(**args_to_dict(args, model_and_diffusion_defaults().keys()))
    model.to(dist_util.dev())
    
    # 创建专门用于推理采样的扩散对象（固定为 ddim 或其他步数）
    args.clip_denoised = True
    args.use_ddim = True
    args.timestep_respacing = "ddim20"
    _, sample_diffusion = create_model_and_diffusion(
        **args_to_dict(args, model_and_diffusion_defaults().keys())
    )

    # 5. 加载数据
    data = yield_dataloader(get_dataloader(
        root_folder_data=args.root_folder_data_train,
        folder_Mo=args.folder_Mo_train,
        folder_P=args.folder_P_train,
        Path_inverse=args.Path_inverse,
        num_images=args.num_images_train,
        batch_size=args.batch_size,
        image_size=args.image_size,
    ))
    
    # 用于验证的固定样本
    dataloader_sample = get_dataloader(
        root_folder_data=args.root_folder_data_sample,
        folder_Mo=args.folder_Mo_sample,
        folder_P=args.folder_P_sample,
        Path_inverse=args.Path_inverse,
        num_images=args.batch_size, # 取一个 batch 用于验证显示
        batch_size=args.batch_size,
        image_size=args.image_size,
        shuffle=False
    )
    sample_Mo, sample_Mr, sample_P, _, _ = next(iter(dataloader_sample))

    # 6. 启动训练循环
    logger.log("training...")
    training_loop = TrainLoop(
        model=model,
        diffusion=diffusion,
        sample_diffusion=sample_diffusion,
        Path_inverse=args.Path_inverse,
        weight_path_similarity=args.weight_path_similarity,
        sample_Mo=sample_Mo,
        sample_Mr=sample_Mr,
        sample_P=sample_P,
        data=data,
        batch_size=args.batch_size,
        microbatch=args.microbatch,
        lr=args.lr,
        ema_rate=args.ema_rate,
        log_interval=args.log_interval,
        save_interval=args.save_interval,
        resume_checkpoint=args.resume_checkpoint,
        use_fp16=args.use_fp16,
        fp16_scale_growth=args.fp16_scale_growth,
        schedule_sampler=create_named_schedule_sampler(args.schedule_sampler, diffusion),
        weight_decay=args.weight_decay,
        lr_anneal_steps=args.lr_anneal_steps,
    ).run_loop()
    logger.log("training complete")

    # === 核心清理逻辑 ===
    # 1. 删除包含大量显存占用（模型参数、梯度、优化器状态）的对象
    del model
    del diffusion
    del sample_diffusion
    del training_loop
    del data
    del dataloader_sample
    
    # 2. 强制进行垃圾回收
    gc.collect()
    
    # 3. 清空 PyTorch 的显存缓存池
    if th.cuda.is_available():
        th.cuda.empty_cache()

Starting Experiment: use_MFF_MAC=True, Path_inverse=True
Logging to /home/pengs/DiffRP_IDDPM/my_model_checkpoints/2026-04-24-14-21-18-P2_0.001-path_simil0-use_MFF_MAC-True-PATH_2PIXEL-path_inv_True-img128-batch_8-epochs60000
成功加载 12000 对图片,M_o的尺寸为 (C, H, W): torch.Size([1, 128, 128]);M_r的尺寸为 (C, H, W): torch.Size([3, 128, 128]);P的尺寸为 (C, H, W): torch.Size([1, 128, 128])
成功加载 8 对图片,M_o的尺寸为 (C, H, W): torch.Size([1, 128, 128]);M_r的尺寸为 (C, H, W): torch.Size([3, 128, 128]);P的尺寸为 (C, H, W): torch.Size([1, 128, 128])
training...
-----------------------------------
| F1                   | 0.0215   |
| F1(EMA)              | 0.0377   |
| grad_norm            | 0.116    |
| loss                 | 0.0184   |
| loss_q0              | 0.022    |
| loss_q1              | 0.0199   |
| loss_q2              | 0.017    |
| loss_q3              | 0.0145   |
| mse                  | 0.0184   |
| mse_q0               | 0.022    |
| mse_q1               | 0.0199   |
| mse_q2               | 0.017    |
| m